# VAE Part 2 of 2: a reparameterized sample that gradients can pass through

This continues **u18-p01**. The encoder outputs `mu` and `logvar`; the decoder needs a sample $z \sim N(\mu, \operatorname{diag}(e^{\text{logvar}}))$ that keeps the computation graph, so that the loss of u18-p01 can train the encoder.

```python
def reparameterize(mu, logvar):
```

`mu` and `logvar` are float tensors of the same shape (usually `(B, k)`). Return

$$z = \mu + e^{\text{logvar}/2} \odot \varepsilon, \qquad \varepsilon \sim N(0, I),$$

a tensor of the same shape and dtype as `mu`.

**Random numbers.** Draw $\varepsilon$ with exactly one call, `torch.randn_like(mu)`, and no other random calls. The grader sets the seed before calling you and reproduces the same $\varepsilon$.

**Gradients.** When `mu` and `logvar` require gradients, `z` must too, and backpropagation must give $\partial z/\partial\mu = 1$ and $\partial z/\partial\,\text{logvar} = \frac12 e^{\text{logvar}/2}\varepsilon$ elementwise. Do not modify the inputs.

**Example.** After `torch.manual_seed(0)`, `torch.randn_like(torch.zeros(2))` is `[1.5410, -0.2934]`. So with that seed, `reparameterize(torch.tensor([1.0, -2.0]), torch.tensor([0.0, 2.0]))` returns `[2.5410, -2.7976]` (the second standard deviation is $e^{1} = 2.718$).

**Restrictions.** Only `torch`. You may not use `torch.distributions`, `torch.normal`, `normal_` or `torch.no_grad`.

**Constraints.** Up to $10^6$ elements, float32 or float64. Time limit 0.5 seconds.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem2_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem2_submission.py
import torch


def reparameterize(mu, logvar):
    # Your code
    return z


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem2_submission as submission
importlib.reload(submission)
# e.g. submission.reparameterize(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9Wwlz2ziW/itYddWYzMiMrBzbpcS966TdPdnKVXFmZ7YsFxsSIZljimQTZORj/N/3ew8ACUqyE6dnN6myRBB4ePcF6GaQJoOJGDQHP+6Xo/FgKAYrVUsM3Wy/qdM6UzT230fH4qOsajEWxUKMJ0KKSpWyklirqvRaJULLVZkpUZ/LWiwrmaQqr7WYy1yUUmuMV0WzPCeomdK6yN1WmUUizZtaaYwejPCYpItFOm+y+gojTzAAaBV9H/T35aVF0mQqzjFIE8qqmGVqNY51M1ulWqfYC7NklhVrlcTpqiyqmjY6HdRFNT8fnOHtTOa5SngwSXVdpbOmxkJNK/OiWsms+xabrzFR6RbHWVGUBHQhM62IdUrXcZ2uVNHUMb0YE1m1XJqdlyqXafRFMgHmoUeYrC3ajGJUyTzBHumF6sZkUxctCrpoqjmTX1TpMs0NvgBYjzFYV426xfMizZjFN4NzWeUQQ1QyT6f8/y8qK1WljQgtF1mUNGjYJoLfFlWxEna5G330WxhN82l+ZKeLFJIv8hpCe9zkaa0fv6SPnx5bqHgu0+Snx2Y2kIjEm1okapECqiDWiUWTz1kCguSa8GD8yOBWywtSNDXNOxELowXiNwz9Fonjvx+9+/j2+ERkkCaRpAxYAqaFPi/WucCiT00ugi+pToFV+EKcELhaVA22VV9UdcWLQNovMs2aCisrmWol6FGs0/ocdgB4Wi6VWFdpDYLFoqjEK5VPYBBXYk3orqXG2yJfDkVOUEXSrEpxniYJpqd52dSauGekkFuWzovyqn0gPSL+2scc668EoOYljf4gXheZnInXH/+qhayU0KTpFfEqJ1T2tHg3Fh+rYih0IbJiLjOCmOZLLVYNmDJTxFSwqCCjS/E6S4kPM9B/DlEu5BwqF9FO42hEwpVi2ZD8A7OzBqxM7cPGlYQnKBUEhlls8dkVtCVbCOACLOZFRYyuFDG9FumCECRfQjZp6bBWQ0w5+Xh8/HP8y9Hrzx8+iUPanQie5vOMfAqJITi+nKuSNCWcTHOBf+RuzCwoFDiukgC6mAzFSi/dHOybF8TkPLEj9K8TbkBzOyBxXgaXbi38kLfGSYhM0htdgP4017XM5yq4HJr30WeV66IKveW8raqbKheXUQJHPD8PwmheNvjLYg5CM1kxmeINb3dcVUXlQTEke7DyMpJaVpW8At4dHfpclipYFvUQSpnTX+jn4RQ+VcPVTgcOMw1e1w3ceUCUY34Y8VKLC/MUc9wkghUOxWI6uCGAtxC4NnuJGw2/oy5LNa+hEzfe/Fts12E2zwp9H2ZDUdVFdnig9p8PhbRff3T4wrLWQLlFVur6CoQuskISYjTOe/ZfeMQsDX1E0tp8vYMcQ4CdHu6izb1i8toNIJBUw72lNVYDjywLQgFPQVrov1vbd/725Ellyp4wpzVpvmiBW0UmeWeZZSKYYdlFfyy76I+veKqqwDFmBCG3kpf0IWc6WIp9sQ7DkGCDUARZoRDTxIiMb4ep9PnEjk58kRn8gwgyWS3J7SpSWHGDj0n0ZHkb9mSvfm9k9g1a2TcqM5WQzhPW9GHIHAUSsq4r+346YHF0QHaoi6cg3Zz/c7XwVYPxjy0fgNyWMflcbSGQVCabGEMZCFsmvoMBZ3yDV/9W9VCjWRjqCSPVMZsIi4O+6TvlwRt6EumWbOFPo2CFBRxGMWdrcdxDZwn3R5Kzu+65SXsOah/TWOeyRCSv/39dc1bk6oFOGVs6rbwkleTYPggREhO23j4+QZLOwUhKXEIfnz5xPewIYJAoVR5+RqbnKZhF8DPYt4neNoCwF0JoKCKY/O6yx3iIJZBDMft+vstv4ruZYuyC9/tetktiO8PRfVdgN5JmEx3Mwl2+Bss7P7NzeesnZvYry7Zv2UTB0Hg7aHbuy8rBYQhe8lLEq6bmWiBY5EPxCO4UlvHo0cWavjlMTOr4GhFALHLeeUEJKtGAHAdZ4BLmJXOkjNWyWSEpRwgRZSbnKnJZJ0d9sif4xdPOsmTI+aykBbTjmZmJ7IyCRx5sImRe05IUigyCdUhLFVIZVUnEt+u0DMwa3qOn4OxPWuWyTuQKhQ10MVGU4KeLlDJLSpk7Wn64ScWfxcHtC7EuqgvK6iUrryDxIR9t/aVlMpDvOExJfJoHWlEqCKy+lcsvfBbDu/op945MmotPxEON3BsUEIM4191rVyJ5NzUUbfUBICoqwVAgrTSRxHn4OznngpqWgEiTdVO1wok9JQeoLvZnTQJPSsOcgyMtT6mAakqYk6jX6VwRQHAoRYQuFtaCgbPjqGYVopEF8iUK3wQBKxq8i8RnepFWGAYx2R5tQXHBbEYcVklPreoRZZOAEKG6XMQMRlXO23xFlZJ692LkKLVNRyy9h8IKUTwWftnQqWRMqliRNQRjX+8gRGzz8tBC2nBFM9Q0F92QHA/FxRi79f0jozzcGOxR8nVWMJ7EiDGzYewNMxtQsQUJhdg72OHnmz5BbEd1UbDeoPhP6kk0Xtwiy4eWqRe26mqVmJUJZseARJNTPX9jWWPXcQ2Jws6JeMc/3hKVJcrgBVRrSQmsKTJvrJwYEFnHvQZa5UtYp0qcwLoCh3oixQrcXkioX9xOnHLfpNFSpqZHo3pdjl9pBBYAzLrmQcpO4zzNErK6OXVGUDCnZAs0879OPrwHrjlbjq4TYMi2Wl7V5xjZfyPcPuKlbXHESVr9JF52e+AlnmGV/+Sx+qetkh/G5pX/EOxlnaWzdsh8YCRq6jTrhov26z900TUJSlmf+8t1usxlt0xf6Y32gnuoEBdmcn7RjkCLc2oWmF6DnqcgM1etu8PARVrvZwrTxEH0PDqwOZ2eUJ9oRe2+IbsUVBc2+DjnV8oUXF5Oc7dHtEgzaLR7DKYDoF1UiipA22lBJho9+mt+kVMPRxcZdVQK7gBgw7QkqUWPbKb4l+NPx7Acy4voIz6DOKY2WByHERwXLUdWV8oKmE1zcCWiyRG0VVV1MIIxIH0gMCEA7up9cS9I/IBY/buciOOnozF3wt7+7eh/TuJXR+/fH/8MDG6mg9i2HuOYiAFhTp7mUV2quf2GPN98mxerErhOB7fT/O2HDx/j9x9+Pj4BuAC6Ev1SwFbpy9+guMp8PdJX+bx98RY55GvAME8nynv4GUlm9/Sryik+F9XxZRn6LZbPphfztS5LLFHprQLSsWYFf0N9zNZiuVR0gPxuBGU283h+ruYXOtDVnERcy368/WTsnbJhaj2Dj8WFYc5KL8GYF3CLlAGaHh+bja73WulwMmRbiRSo6io1bcVeiDJB6PRsVzZbVwqJHXMJWoL6Goj289CTK9Tnl5yHUlNObeeHpxbtCbSF28MW+wm5Sc3LbYlc5MbP3KiIPvPidkLfMRvFz61F0LayKSSAXxEcM+xko79NPDo9s4jOZDvOcbJ2IYfboXGjGZg3PoM9xKZXasdJxZEywnbTBKZCmVhlEgUDGBDM/IAqGkq6qYWuyMVfcO6msogU227A7XKP5ytZr5rMH1lwdwSumnwzmL+W2UVAwtgI3V6aTtONPpvSYLOqaFNZmhgxupPtAGZbKsiXA8mTKClfUWvBPEbYDYykzDFYIfOEO6JijoCvGFkjiHCyOzZ6sohkktg9wu3JnWy8eZEus5TETXue7h+c7VjYCS9qyoSy7h1r/SIxu5+Nv8DrbVIDgUJWzEfbeAf5nUHt4CWvsIzE9z/AxU0OAtoGFzzW9TiwSwM2lt7FvO9aynQ+lOlHznY2ad9QCMaDDO2boL73PPJ9ANPkbqp2T9q1YxesNvdkw49kWao8CVzHxyyaDozLM+79P/r82rULR8FXaf7BRrGjZnmkKQaFpreysSAq7Lx3sn6HlHFLp40PeiByQMxTSA8kJYkW1J3u32UBxkqMytMoGwTNi/5RUHnKDjbw9gnD3d7lzxRRwBgH6mZvKPYMEGdRZIB72A9173KPWrG3u/uJu/CnlsUO9LXDvQP2Q3uCOJO5Jt7Oz22E4FSwCxDcflYSRezf+TN8YarRbC2v9L45mEUlg7wwbQGRUlpoRZ5dDd2eqFy6UBMYLTBNJJtKUdJxJTK1RGm+prRJ8MwoikIvmCEpE0lBQjGAz1OuL40UAi9o/omDoxeFDbo2+EIN/ykCL5b+SfTyQk+FsMODVQcYaGH5Q/ADHe7QHUAO/4iE88Lt0R7JdlIG5n3iY2NEcVGafLLfbaPpxsg2TO9bCKaKEnrxn8LBppLVsxsRmMzJ7nA6OvOVewf5D2FBt+t9xLNz2yKZR7+PYJMmsbL3CMUwmTEpdaXOVc5FbJs78qp/Lf2Wstt7KvVFlQJMdhVzJhsABArdmKqpfkb/IVf7jGei9LxKuaoQJVQV9r0U9iSaJ6Tu+JoKNpironmaD+v7HaYZtVVc3Rqhbsb3Oq5ngYriuH2Bks8mmwQcmeaCg/qC9gEISGpB5adyOR9Vfi0JNitdU8uEGlbkZbk3aZUOfykd87L23hCJghWGZvNxWoe+ZSadmPAJifLORyj7v73hXW9pvmP2Cpl40NZi8LSbVS5VsrJafjltU0RHyp0zx2deSWw5RQn4oWinPHHpOvQe49RwiLJCJjoIGInHpDZkE9yKGBA8xCzqZATO2aF8IogWGX+C4wZ3Ew/Ftl2YgpGGTs/omUo93T2axoxRWVtJkybSwPsCIvDhn3bQzgidO0rR1tApgyUT1KcGqzOOFvTEVnLmp7OupHP5rOa+49a2vpfgrkXA7KTLITow0/0ob7TEtTwXQDotopOa2idvPjjmmSZPZD4C+3Ty5tejt5/eDW2VbqfCncXc3HWnwJ4v619eIl6ORw6XfnXMl2C6ZhXEmaSVgvEZYQTAdDPNcjiqOqUtKofmm89v3h1/ij8dH70dtthtpDm6VKQ9/VZYRKMx9WZMZ4e6k3xk4lHk3REzqtE15iAmz1Vt7NfMtrezsHhD2jqgP5sLYTFmnj6l11wwnBnF3yaJbUhVEfWAYrOMPE/4HYwb9XsTtu3Sa0lYRbTGcWa8maGRPDB7Nfa4dVFc8JmCuHHymEQj6gA712V3eSW1ajtEpgfStcRevT0ejQ46BB5Exl0Y24MhRpNOYuiGWGEFZZOge+JRa9ZboHcxyvkV2hf6TBrFJ/EBCvCno9Focvbdhsy3Gvt9HB7y73eYOe6s2J1ogpaheRV+Xbgdr5JCmaTJXLZDdCEQt37d/kASlg+zyenAXPojK2xDRnsPsKXb3iP8uu0tPeNb3mVLBpqdZa/72caWq/LMFPIM7uqgS+Kt26NYQ0lDgEJlYRw7fSPf/kVW2u0RpbVa6YBvz+T9qMBe1Z370wGKBBrBIvQbXvASdAjJKQntuJE8czgm7a8a+C2jFJSt8C2C3JG24XDJPad5o3z5zU2ANf5wwkBOn03O6BIqnQsDW9ulFuwepwML2k32dqO31A2msmdXmWL9rTn0oaFRNLp9yJnY9sWH74g6fzTy2MO5ba/8XS7NCuG04xxZKiXe23c3tjx4t9jkHsbG2VlnRb505zKbPlsEaW5umXE2P6QCYiEr4U4GQ98J2M35cGOjp/0HCW5xpvR6x12VB8WSfxUu/j0DPq9IvjGGdLBaBSd4FbQ4Ce46qB2KJ7vCm01nz1xBVrWO7cGhqF1gctVDKh2DLne9LytlpzIriizYQsy8I1B1T3MJXN0D165wBcqdIYWvNS+EK3iMd4tjqm/ooGzijgeo3DEnu22woDNd2z5DzX/enWCai0d3HdYN+WB7aK6cDv07NkN7HYRwai+Qw+Vb160u+ScOIMu7u2hvrwWrZgizWiIUQEe882rTfVrJnG79aL44aw6pWeX5DGTzij9g9cvsVdPeP4NO2ktRl2Uwip6JR3bXdkaIIcAd0h/vfi0rAOUM34WoOSDzryMBVv8XC8oHHe6+IQgwG/e+KDS0l5/4DeIevelfnzWXKKlZYa5RdnePwRszxLccrrcvZXaLdlzLbFdvXNelRQlfWqQji8Z893Yw727aaT3Qbv6tCJJKrlnKHLF2CTq89/aDr3cce5w9tJxu9cewLTg9iOBf9sfR6Cwcbrwa0St+40t1t2o4h2nuFLPgdur6KDwdnZnza8ZyIq7929rPutvaz8MNusx1Vp8s9ktDK7/evbHT4Cl5zaEIDobigD+fD8WPYa+KXrbMaM+zg7Cn0gejESzIP7hZNX0LDOw136WDcLgcGnEfmllcLD9/6oHYEMTDwMBaxx0ow+07jDW8QwTaiADKaX9TMkcIFTeab8WJwO20IRn6mZUTzcF4UzbtL7h88Xydv//u+kabXIUePP0GbsCl/N4gldOMQBx4VyR3cvmPg72+0wKe+A7hug+CFP66K6bsO65aJty/tL+Go99BNXRxgdqjCEfgC4VQs0Vr+euWqgyusqT8e5/UfMg/GruDIPNbDmKAhRKsoUvXYaSbFaRCDc+1rBKXSMdDG212qlCPVvgvIsS116l35mG99W5oOsROY8AMxCH4QdAKc94g1Si43YF/qGCqP17JThKUEYJUzq+aexW2g+dhRlf7HwkTGb81Ut6DgkP/W+3G//Fg32G3Ml400LZgPKJ/Q9j0QfRst4pvTqQkB9XtEtSMn4Vf1d+DA1+o9MuOaz5fC0yNyrko7S1eUkV24G5oYRayTH/SqJs09LJourrCWq4REr8oQbDFzcYek+gJag/uldYJvfVh88sXXeQENjwVG/Zu+jNn6S7qV0LgtaoKTT4eIqK/bQDcenFXyuPe2GvBQIQz/vszncHt7f8COCcwxw==")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
